# Figure 2 — submission notebook

Plot-only, local-data-only reconstruction of authoritative panels A–H.

## Plot-only execution contract

This notebook renders existing intermediate numerical results only. It does not recompute upstream analysis.


In [ ]:
PLOT_ONLY_FIGURE = "Figure_2"
ANALYSIS_RECOMPUTED = False
ALLOWED_INPUT_SUFFIXES = {".h5ad", ".h5", ".csv", ".gz", ".parquet", ".json"}
print(PLOT_ONLY_FIGURE, "plot-only mode; analysis_recomputed=", ANALYSIS_RECOMPUTED)


In [ ]:
# Conventional Jupyter inline plotting: live Matplotlib figures are rendered by plt.show().
get_ipython().run_line_magic('matplotlib', 'inline')

# Figure 2 — independent reviewer notebook

This notebook generates every main Figure 2 panel from the adjacent `data/Figure_2/` bundle. Supplementary Figure 2 embeds immutable, checksum-validated raster derivatives packaged in the same bundle; it does not access upstream analysis directories. Frozen UMAP coordinates are treated as immutable analysis results; ARI and all plot annotations are recomputed from the bundled labels.

In [ ]:
from pathlib import Path
from functools import lru_cache
import hashlib, importlib.util, json, math, re, sys
import numpy as np
import pandas as pd
import h5py
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score
from IPython.display import display

# ─── Figure parameters (user-editable) ───────────────────────────────
FONT_FAMILY = ["Arial", "Helvetica", "DejaVu Sans"]
PANEL_LABEL_SIZE_PT = 10; AXIS_TITLE_SIZE_PT = 9; TICK_LABEL_SIZE_PT = 7
LEGEND_SIZE_PT = 7; ANNOTATION_SIZE_PT = 7; PAGE_TITLE_SIZE_PT = 12
COMPARATOR_POINT_SIZE = 0.18; SCKNIFE_POINT_SIZE = 0.10; UMAP_ALPHA = 0.75
HALLMARK_DIAGNOSTIC_POINT_SIZE = 3.0; HALLMARK_DIAGNOSTIC_ALPHA = 0.67
RASTERIZED = True; OUTPUT_DPI = 300
A4_LANDSCAPE_FIGSIZE = (11.69, 8.27); OVERVIEW_FIGSIZE = (11.69, 6.2); METRIC_HEATMAP_FIGSIZE = (10.5, 3.3); TUNING_FIGSIZE = (11, 8); METRIC_CMAP = "RdBu_r"
# ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({"font.family":"sans-serif","font.sans-serif":FONT_FAMILY,"pdf.fonttype":42,"ps.fonttype":42,"axes.grid":False})
# Restrained two-colour gradients: darker tones echo the benchmark palette.
HALLMARK_BGNMF_DARK="#7A1F3D"; HALLMARK_SCANPY_DARK="#1F5D70"
HALLMARK_BGNMF_DOT_DARK="#55162B"; HALLMARK_SCANPY_DOT_DARK="#16414E"
HALLMARK_BGNMF_CMAP=LinearSegmentedColormap.from_list("hallmark_bgnmf",["#F5E7EB",HALLMARK_BGNMF_DOT_DARK])
HALLMARK_SCANPY_CMAP=LinearSegmentedColormap.from_list("hallmark_scanpy",["#E4F0F2",HALLMARK_SCANPY_DOT_DARK])
METHOD_ORDER=["scKNIFE","SPECTRA","ExpiMap","Slalom"]
METHOD_SLUG={"scKNIFE":"scknife","SPECTRA":"spectra","ExpiMap":"expimap","Slalom":"slalom"}
METHOD_LABEL={value:key for key,value in METHOD_SLUG.items()}
METHOD_COLORS={"scKNIFE":"#A46BBE","SPECTRA":"#63B8D0","ExpiMap":"#E76F70","Slalom":"#43AAA5"}
DATASET_ORDER=["bassez_c1","bassez_c2","wu_tnbc"]; SCKNIFE_DATASET_ORDER=["bassez_c1","bassez_c2","shiao","wu_tnbc"]
DATASET_LABEL={"bassez_c1":"Bassez C1","bassez_c2":"Bassez C2","wu_tnbc":"Wu TNBC","shiao":"Shiao"}
UMAP_COLUMNS=["cell_id","umap_1","umap_2","leiden_cluster","author_cell_type"]
CATEGORY_PALETTE=["#0072B2","#E69F00","#009E73","#CC79A7","#56B4E9","#D55E00","#F0E442","#6B6B6B","#8DD3C7","#BEBADA","#FB8072","#80B1D3","#FDB462","#B3DE69","#FCCDE5","#BC80BD","#CCEBC5","#FFED6F","#1B9E77","#7570B3"]
DATA_DIR=(Path.cwd()/"data"/"Figure_2").resolve()
if not DATA_DIR.is_dir(): raise FileNotFoundError("data/Figure_2 must be below the main_figure notebook directory")
manifest=json.loads((DATA_DIR/"manifest.json").read_text())
if manifest.get("format_version") != 2: raise RuntimeError("Figure 2 requires data/Figure_2 manifest format_version 2")
expected_umaps={*(f"{method}_{dataset}_umap.csv.gz" for method in ("expimap","slalom","spectra") for dataset in DATASET_ORDER),*(f"scknife_{dataset}_umap.csv.gz" for dataset in SCKNIFE_DATASET_ORDER)}
expected_inputs=expected_umaps|{"runtime.csv","knowledge_feature_counts.csv","agreement_metrics_from_frozen_labels.csv","tuning_sweeps.csv","tuning_final_parameters.csv","tahoe_plot_data.h5"}
manifest_names={entry["name"] for entry in manifest["files"]}
if not expected_inputs.issubset(manifest_names): raise RuntimeError(f"Manifest inventory is missing inputs: {sorted(expected_inputs-manifest_names)}")
def sha256(path):
    digest=hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b""): digest.update(chunk)
    return digest.hexdigest()
validation_rows=[]
for entry in manifest["files"]:
    source=DATA_DIR/entry["name"]
    if not source.is_file(): raise FileNotFoundError(source)
    if source.stat().st_size!=entry["bytes"] or sha256(source)!=entry["sha256"]: raise RuntimeError(f"Size/checksum mismatch: {source.name}")
    if source.name.endswith("_umap.csv.gz"):
        frame=pd.read_csv(source,dtype={"cell_id":str,"leiden_cluster":str,"author_cell_type":str})
        if list(frame.columns)!=UMAP_COLUMNS: raise ValueError(f"{source.name}: invalid columns")
        if frame[UMAP_COLUMNS].isna().any().any() or frame.cell_id.duplicated().any(): raise ValueError(f"{source.name}: invalid rows")
        if not np.isfinite(frame[["umap_1","umap_2"]].to_numpy()).all(): raise ValueError(f"{source.name}: non-finite UMAP coordinates")
        validation_rows.append({"file":source.name,"cells":len(frame),"clusters":frame.leiden_cluster.nunique(),"author_types":frame.author_cell_type.nunique()})
        del frame
print(f"Validated {len(manifest['files'])} inputs and {len(validation_rows)} UMAP schemas/checksums (matplotlib fallback; cnsplots unavailable in Python 3.9 environment).")
display(pd.DataFrame(validation_rows))
TAHOE_BUNDLE = DATA_DIR / "tahoe_plot_data.h5"
TAHOE_EXPECTED = {"FAS": 64064, "TP53": 64064, "BIRC5": 64064, "ICAM1": 64064, "PIK3CA": 60743, "PIK3R1": 60743, "RPTOR": 60743}
tahoe = h5py.File(TAHOE_BUNDLE, "r")
if set(tahoe) != set(TAHOE_EXPECTED): raise RuntimeError("Unexpected Tahoe Hallmark bundle groups")


## Derived benchmark and tuning tables

Runtime and capacity use compact local summaries. ARI, AMI, and NMI are recomputed on the exact shared-cell intersection. Hyperparameter pages use the original single-seed tuning summaries and distinguish each data-derived optimum from the parameter value used by the production pipeline.

Selection uses maximum silhouette for `k`, joint `k × lambda`, and `beta_H`. For `beta_W`, the maximum-silhouette value within W sparsity 0.35–0.55 is selected; when no value passes, the closest value to that range is used. With one seed per sweep, no uncertainty band is shown.

In [ ]:
@lru_cache(maxsize=4)
def load_umap(method,dataset):
    return pd.read_csv(DATA_DIR/f"{method}_{dataset}_umap.csv.gz",dtype={"cell_id":str,"leiden_cluster":str,"author_cell_type":str})
runtime_df=pd.read_csv(DATA_DIR/"runtime.csv")
if list(runtime_df.columns)!=["method","dataset","seconds"]: raise ValueError("runtime.csv must contain method,dataset,seconds")
runtime_df["method"]=runtime_df.method.replace({"bgNMF":"scKNIFE","Spectra":"SPECTRA"}); runtime_df["hours"]=pd.to_numeric(runtime_df.seconds,errors="raise")/3600
expected_runtime_keys={(method,dataset) for method in METHOD_ORDER for dataset in ("Bassez c1","Bassez c2")}
if runtime_df.duplicated(["method","dataset"]).any() or (runtime_df.seconds<=0).any() or set(zip(runtime_df.method,runtime_df.dataset))!=expected_runtime_keys: raise ValueError("Invalid or incomplete runtime rows")
capacity_df=pd.read_csv(DATA_DIR/"knowledge_feature_counts.csv")
capacity_order=[("SPECTRA","Benchmarking"),("ExpiMap","Benchmarking"),("Slalom","Benchmarking"),("scKNIFE","Benchmarking graph"),("scKNIFE","Full in-house graph")]
if list(capacity_df.columns)!=["method","scope","feature_type","count"] or (capacity_df["count"]<=0).any() or capacity_df.duplicated(["method","scope"]).any() or set(zip(capacity_df.method,capacity_df.scope))!=set(capacity_order): raise ValueError("Invalid or incomplete capacity table")
capacity_df=capacity_df.set_index(["method","scope"]).loc[capacity_order].reset_index()
metric_rows=[]
for dataset in DATASET_ORDER:
    frames={method:load_umap(METHOD_SLUG[method],dataset).set_index("cell_id") for method in METHOD_ORDER}
    shared=sorted(set.intersection(*(set(frame.index) for frame in frames.values())))
    for method in METHOD_ORDER:
        frame=frames[method].loc[shared]; truth=frame.author_cell_type; labels=frame.leiden_cluster
        metric_rows.append({"method":method,"dataset":dataset,"n_shared":len(shared),"n_clusters":labels.nunique(),"n_author_types":truth.nunique(),"ari":adjusted_rand_score(truth,labels),"ami":adjusted_mutual_info_score(truth,labels),"nmi":normalized_mutual_info_score(truth,labels)})
agreement_df=pd.DataFrame(metric_rows); reference_df=pd.read_csv(DATA_DIR/"agreement_metrics_from_frozen_labels.csv")
key=["method","dataset"]; cols=["n_shared","n_clusters","n_author_types","ari","ami","nmi"]
observed=agreement_df.sort_values(key).reset_index(drop=True); reference=reference_df.sort_values(key).reset_index(drop=True)
if observed[key].to_dict("records")!=reference[key].to_dict("records") or not np.allclose(observed[cols],reference[cols],rtol=0,atol=1e-12): raise RuntimeError("Shared-cell metrics differ from frozen reference")

tuning_df=pd.read_csv(DATA_DIR/"tuning_sweeps.csv")
tuning_final_df=pd.read_csv(DATA_DIR/"tuning_final_parameters.csv")
required_tuning={"dataset","phase","k","lambda_reg","beta_W","beta_H","silhouette","recon_corr_scknife","sparsity_W","tuning_selected","production_used"}
if set(tuning_df.columns)!=required_tuning: raise ValueError(f"Invalid tuning_sweeps.csv columns: {tuning_df.columns.tolist()}")
if set(tuning_final_df.dataset)!=set(["Bassez C1","Bassez C2","Wu TNBC","Shiao"]): raise ValueError("Unexpected tuning datasets")
if tuning_df.groupby(["dataset","phase"]).tuning_selected.sum().ne(1).any(): raise ValueError("Every tuning phase must have one data-derived selection")
if tuning_df.groupby(["dataset","phase"]).production_used.sum().ne(1).any(): raise ValueError("Every tuning phase must have one production-used value")
load_umap.cache_clear()
display(runtime_df.sort_values(["dataset","method"])); display(capacity_df); display(agreement_df); display(tuning_final_df)

## Figure 2A–E — workflow, compact benchmarks, agreement heatmaps, and scKNIFE tuning

In [ ]:
def clean_axis(ax):
    ax.grid(False); ax.spines[["top","right"]].set_visible(False); ax.tick_params(labelsize=TICK_LABEL_SIZE_PT)
def draw_workflow(ax):
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off"); ax.set_title("A  Benchmarking workflow",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold")
    stages=[(.02,"Cohorts","Bassez C1 · Bassez C2\nWu TNBC · Shiao*","#EAF2F8"),(.27,"Methods","scKNIFE · SPECTRA\nExpiMap · Slalom","#F4ECF7"),(.52,"Frozen results","UMAP coordinates\nLeiden + author labels","#E8F6F3"),(.77,"Evaluation","Runtime · capacity\nARI · AMI · NMI","#FEF5E7")]
    width,height,y=.20,.55,.22
    for index,(x,heading,detail,color) in enumerate(stages):
        ax.add_patch(FancyBboxPatch((x,y),width,height,boxstyle="round,pad=0.012,rounding_size=0.018",linewidth=1,edgecolor="#4D4D4D",facecolor=color)); ax.text(x+width/2,y+.37,heading,ha="center",va="center",fontsize=AXIS_TITLE_SIZE_PT,fontweight="bold"); ax.text(x+width/2,y+.18,detail,ha="center",va="center",fontsize=ANNOTATION_SIZE_PT,linespacing=1.4)
        if index<len(stages)-1: ax.add_patch(FancyArrowPatch((x+width+.008,y+height/2),(stages[index+1][0]-.008,y+height/2),arrowstyle="-|>",mutation_scale=11,linewidth=1.1,color="#555555"))
    ax.text(.02,.06,"*Shiao is displayed for scKNIFE only; common-dataset metrics use shared cells from Bassez C1, Bassez C2, and Wu TNBC.",fontsize=ANNOTATION_SIZE_PT,color="#555555")
def draw_runtime(ax):
    datasets=["Bassez c1","Bassez c2"]; x=np.arange(len(datasets)); width=.78/len(METHOD_ORDER)
    for index,method in enumerate(METHOD_ORDER):
        rows=runtime_df[runtime_df.method.eq(method)].set_index("dataset").reindex(datasets); values=rows.hours.to_numpy(); offsets=x+(index-(len(METHOD_ORDER)-1)/2)*width
        bars=ax.bar(offsets,values,width*.90,color=METHOD_COLORS[method],edgecolor="#333333",linewidth=.5,label=method)
        for bar,hours in zip(bars,values):
            minutes=int(round(float(hours)*60)); ax.annotate(f"{minutes//60}h {minutes%60:02d}m",(bar.get_x()+bar.get_width()/2,hours),xytext=(0,4),textcoords="offset points",ha="center",fontsize=ANNOTATION_SIZE_PT)
    ax.set_xticks(x,datasets); ax.set_ylabel("Training time (hours)",fontsize=AXIS_TITLE_SIZE_PT); ax.set_ylim(0,runtime_df.hours.max()*1.22); ax.set_title("B  Training runtime at 20k cells",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold"); clean_axis(ax); ax.legend(loc="upper center",bbox_to_anchor=(.5,-.16),ncol=2,frameon=False,fontsize=LEGEND_SIZE_PT)
def draw_capacity(ax):
    labels=["SPECTRA","ExpiMap","Slalom","scKNIFE\nbenchmark","scKNIFE\nfull in-house"]; colors=[METHOD_COLORS["SPECTRA"],METHOD_COLORS["ExpiMap"],METHOD_COLORS["Slalom"],"#C7A4D4",METHOD_COLORS["scKNIFE"]]; x=np.arange(len(capacity_df))
    bars=ax.bar(x,capacity_df["count"],color=colors,edgecolor="#333333",linewidth=.6,width=.62); ax.set_yscale("log"); ax.set_xticks(x,labels); ax.set_ylabel("Prior-knowledge features (log scale)",fontsize=AXIS_TITLE_SIZE_PT); ax.set_title("C  Prior-knowledge capacity",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold")
    for bar,value in zip(bars,capacity_df["count"]): ax.annotate(f"{int(value):,}",(bar.get_x()+bar.get_width()/2,value),xytext=(0,4),textcoords="offset points",ha="center",fontsize=ANNOTATION_SIZE_PT,fontweight="bold")
    ax.text(0,-.24,"Comparators: gene sets/factors · scKNIFE: unified graph nodes",transform=ax.transAxes,fontsize=ANNOTATION_SIZE_PT,color="#555555"); clean_axis(ax)
def make_overview_page():
    fig=plt.figure(figsize=OVERVIEW_FIGSIZE,layout="constrained"); grid=fig.add_gridspec(2,2,height_ratios=[.36,.64]); draw_workflow(fig.add_subplot(grid[0,:])); draw_runtime(fig.add_subplot(grid[1,0])); draw_capacity(fig.add_subplot(grid[1,1])); fig.suptitle("Figure 2 — benchmarking overview",fontsize=PAGE_TITLE_SIZE_PT,fontweight="bold"); return fig
METRIC_SPECS=[("ari","ARI"),("ami","AMI"),("nmi","NMI")]
def make_metric_heatmaps():
    shared_n=agreement_df.groupby("dataset").n_shared.first().to_dict(); fig,axes=plt.subplots(1,3,figsize=METRIC_HEATMAP_FIGSIZE,sharey=True,constrained_layout=True)
    for panel,(ax,(metric,label)) in enumerate(zip(axes,METRIC_SPECS),start=1):
        matrix=agreement_df.pivot(index="method",columns="dataset",values=metric).reindex(index=METHOD_ORDER,columns=DATASET_ORDER); image=ax.imshow(matrix.to_numpy(),cmap=METRIC_CMAP,vmin=0,vmax=1,aspect="auto"); ax.set_xticks(np.arange(len(DATASET_ORDER)),[f"{DATASET_LABEL[d]}\nn={shared_n[d]:,}" for d in DATASET_ORDER]); ax.set_yticks(np.arange(len(METHOD_ORDER)),METHOD_ORDER); ax.set_title(f"D{panel}  {label}",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold"); ax.tick_params(labelsize=TICK_LABEL_SIZE_PT)
        for row in range(len(METHOD_ORDER)):
            for col in range(len(DATASET_ORDER)):
                value=matrix.iloc[row,col]; rgba=plt.get_cmap(METRIC_CMAP)(value); luminance=.2126*rgba[0]+.7152*rgba[1]+.0722*rgba[2]; ax.text(col,row,f"{value:.3f}",ha="center",va="center",fontsize=ANNOTATION_SIZE_PT,color="black" if luminance>.58 else "white",fontweight="bold")
        for spine in ax.spines.values(): spine.set_visible(False)
    fig.colorbar(image,ax=axes,location="right",fraction=.025,pad=.02,label="Agreement score"); return fig
for maker in (make_overview_page,make_metric_heatmaps):
    fig=maker(); display(fig); plt.close(fig)

TUNING_DATASET_ORDER=["Bassez C1","Bassez C2","Wu TNBC","Shiao"]
TUNING_PANEL_COLOR="#365F70"; TUNING_SECONDARY_COLOR="#C26D3A"; TUNING_SELECTED_COLOR="#D73027"; PRODUCTION_COLOR="#111111"

def _format_parameter(value):
    if value==0: return "0"
    if abs(value)<.01: return f"{value:.0e}"
    return f"{value:g}"

def _mark_choices(ax,frame,x_column,y_column="silhouette",categorical_x=False):
    selected=frame[frame.tuning_selected]; production=frame[frame.production_used]
    for subset,marker,color,size,zorder in ((selected,"*",TUNING_SELECTED_COLOR,150,6),(production,"D",PRODUCTION_COLOR,48,7)):
        if subset.empty: continue
        row=subset.iloc[0]; x=(frame.index.get_loc(row.name) if categorical_x else row[x_column]); ax.scatter([x],[row[y_column]],marker=marker,s=size,color=color,edgecolor="white",linewidth=.5,zorder=zorder)

def make_tuning_page(dataset,page_number):
    frame=tuning_df[tuning_df.dataset.eq(dataset)].copy(); final=tuning_final_df.set_index("dataset").loc[dataset]
    fig,axes=plt.subplots(2,2,figsize=TUNING_FIGSIZE,layout="constrained")

    kdf=frame[frame.phase.eq("k")].sort_values("k").set_index(pd.Index(range(sum(frame.phase.eq("k")))))
    axes[0,0].plot(kdf.k,kdf.silhouette,color=TUNING_PANEL_COLOR,marker="o",linewidth=1.5,markersize=4); _mark_choices(axes[0,0],kdf,"k")
    axes[0,0].set_xlabel("Latent factors (k)",fontsize=AXIS_TITLE_SIZE_PT); axes[0,0].set_ylabel("Silhouette",fontsize=AXIS_TITLE_SIZE_PT); axes[0,0].set_title(f"E{page_number}a  k sweep",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold"); clean_axis(axes[0,0])

    joint=frame[frame.phase.eq("joint_k_lambda")].copy(); kvals=sorted(joint.k.unique()); lvals=sorted(joint.lambda_reg.unique())
    matrix=joint.pivot(index="k",columns="lambda_reg",values="silhouette").reindex(index=kvals,columns=lvals)
    global_joint=tuning_df[tuning_df.phase.eq("joint_k_lambda")].silhouette; image=axes[0,1].imshow(matrix.to_numpy(),cmap="RdBu_r",vmin=global_joint.min(),vmax=global_joint.max(),aspect="auto")
    axes[0,1].set_xticks(range(len(lvals)),[_format_parameter(v) for v in lvals]); axes[0,1].set_yticks(range(len(kvals)),[f"{int(v)}" for v in kvals]); axes[0,1].set_xlabel("Graph regularization (lambda)",fontsize=AXIS_TITLE_SIZE_PT); axes[0,1].set_ylabel("k",fontsize=AXIS_TITLE_SIZE_PT); axes[0,1].set_title(f"E{page_number}b  Joint k × lambda",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold")
    for row,kval in enumerate(kvals):
        for col,lval in enumerate(lvals):
            value=matrix.loc[kval,lval]
            if pd.notna(value):
                rgba=plt.get_cmap("RdBu_r")((value-global_joint.min())/(global_joint.max()-global_joint.min())); luminance=.2126*rgba[0]+.7152*rgba[1]+.0722*rgba[2]; axes[0,1].text(col,row,f"{value:.3f}",ha="center",va="center",fontsize=ANNOTATION_SIZE_PT,color="black" if luminance>.58 else "white")
    for subset,marker,color,size,zorder in ((joint[joint.tuning_selected],"*",TUNING_SELECTED_COLOR,180,6),(joint[joint.production_used],"D",PRODUCTION_COLOR,55,7)):
        if not subset.empty:
            choice=subset.iloc[0]; axes[0,1].scatter([lvals.index(choice.lambda_reg)],[kvals.index(choice.k)],marker=marker,s=size,color=color,edgecolor="white",linewidth=.6,zorder=zorder)
    for spine in axes[0,1].spines.values(): spine.set_visible(False)
    fig.colorbar(image,ax=axes[0,1],fraction=.046,pad=.04,label="Silhouette")

    bw=frame[frame.phase.eq("beta_W")].sort_values("beta_W").reset_index(drop=True); ax=axes[1,0]; secondary=ax.twinx()
    ax.plot(bw.beta_W,bw.silhouette,color=TUNING_PANEL_COLOR,marker="o",linewidth=1.5,markersize=4,label="Silhouette"); secondary.plot(bw.beta_W,bw.sparsity_W,color=TUNING_SECONDARY_COLOR,marker="s",linestyle="--",linewidth=1.2,markersize=3,label="W sparsity"); secondary.axhspan(.35,.55,color="#D9EAD3",alpha=.55,label="Target sparsity")
    _mark_choices(ax,bw,"beta_W"); ax.set_xscale("symlog",linthresh=1e-4,linscale=.8); ax.set_xticks(bw.beta_W,[_format_parameter(v) for v in bw.beta_W]); ax.set_xlabel("W sparsity penalty (beta_W)",fontsize=AXIS_TITLE_SIZE_PT); ax.set_ylabel("Silhouette",fontsize=AXIS_TITLE_SIZE_PT,color=TUNING_PANEL_COLOR); secondary.set_ylabel("W sparsity",fontsize=AXIS_TITLE_SIZE_PT,color=TUNING_SECONDARY_COLOR); ax.set_title(f"E{page_number}c  beta_W sweep",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold"); clean_axis(ax); secondary.grid(False); secondary.spines["top"].set_visible(False); secondary.tick_params(labelsize=TICK_LABEL_SIZE_PT)

    bh=frame[frame.phase.eq("beta_H")].sort_values("beta_H").reset_index(drop=True); axes[1,1].plot(bh.beta_H,bh.silhouette,color=TUNING_PANEL_COLOR,marker="o",linewidth=1.5,markersize=4); _mark_choices(axes[1,1],bh,"beta_H"); axes[1,1].set_xscale("log"); axes[1,1].set_xticks(bh.beta_H,[_format_parameter(v) for v in bh.beta_H]); axes[1,1].set_xlabel("H sparsity penalty (beta_H)",fontsize=AXIS_TITLE_SIZE_PT); axes[1,1].set_ylabel("Silhouette",fontsize=AXIS_TITLE_SIZE_PT); axes[1,1].set_title(f"E{page_number}d  beta_H sweep",loc="left",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold"); clean_axis(axes[1,1])

    handles=[Line2D([0],[0],marker="*",color="none",markerfacecolor=TUNING_SELECTED_COLOR,markeredgecolor="white",markersize=12,label="Tuning-selected"),Line2D([0],[0],marker="D",color="none",markerfacecolor=PRODUCTION_COLOR,markeredgecolor="white",markersize=6,label="Production-used")]
    fig.legend(handles=handles,loc="outside lower center",ncol=2,frameon=False,fontsize=LEGEND_SIZE_PT)
    final_text=f"Production: k={int(final.k)}, lambda={_format_parameter(final.lambda_reg)}, beta_W={_format_parameter(final.beta_W)}, beta_H={_format_parameter(final.beta_H)}"
    if dataset=="Shiao": final_text += " · Joint-grid optimum was k=40, lambda=1; production used k=80, lambda=1"
    fig.suptitle(f"E{page_number}  {dataset} — scKNIFE hyperparameter tuning\n{final_text}",fontsize=PAGE_TITLE_SIZE_PT,fontweight="bold")
    return fig

for page_number,dataset in enumerate(TUNING_DATASET_ORDER,start=1):
    fig=make_tuning_page(dataset,page_number); display(fig); plt.close(fig)


## Frozen UMAP panels

The coordinates below are read from the local bundle. Comparator pages pair Leiden clusters with author cell types. The scKNIFE page retains the selected four Leiden-only datasets.

In [ ]:
def natural_key(value):
    return [int(token) if token.isdigit() else token.lower() for token in re.split(r"(\d+)", str(value))]

all_author_types = set()
for name in sorted(expected_umaps):
    values = pd.read_csv(DATA_DIR / name, usecols=["author_cell_type"], dtype=str).author_cell_type.dropna().unique()
    all_author_types.update(values)

def canonical_author_label(label):
    compact = re.sub(r"[._\s-]+", "", str(label)).lower()
    aliases = {
        "bcell": "B cell", "bcells": "B cell", "tcell": "T cell", "tcells": "T cell",
        "cancercell": "Cancer epithelial", "cancerepithelial": "Cancer epithelial",
        "endothelialcell": "Endothelial", "endothelial": "Endothelial",
        "fibroblast": "Fibroblast", "fibroblasts": "Fibroblast", "mastcell": "Mast cell",
        "myeloid": "Myeloid", "myeloidcell": "Myeloid", "normalepithelial": "Normal epithelial",
        "epithelial": "Epithelial", "plasmablasts": "Plasmablasts", "pvl": "PVL", "pdc": "pDC", "cafs": "CAFs",
    }
    return aliases.get(compact, str(label))

canonical_types = sorted({canonical_author_label(label) for label in all_author_types}, key=natural_key)
if len(canonical_types) > len(CATEGORY_PALETTE):
    raise ValueError("Author cell-type palette is too short")
canonical_colors = {label: CATEGORY_PALETTE[index] for index, label in enumerate(canonical_types)}
AUTHOR_COLORS = {label: canonical_colors[canonical_author_label(label)] for label in all_author_types}


def categorical_scatter(ax, frame, column, title, point_size, colors=None):
    categories = sorted(frame[column].astype(str).unique(), key=natural_key)
    color_map = colors or {category: CATEGORY_PALETTE[index % len(CATEGORY_PALETTE)] for index, category in enumerate(categories)}
    for category in categories:
        selected = frame[column].astype(str).eq(category)
        ax.scatter(frame.loc[selected, "umap_1"], frame.loc[selected, "umap_2"], s=point_size,
                   alpha=UMAP_ALPHA, color=color_map[category], label=category, linewidths=0, rasterized=RASTERIZED)
    ax.set_title(title, fontsize=PANEL_LABEL_SIZE_PT); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, markerscale=5, fontsize=LEGEND_SIZE_PT)
    ax.set_aspect("equal", adjustable="datalim")


def make_comparator_page(method):
    fig, axes = plt.subplots(3, 2, figsize=A4_LANDSCAPE_FIGSIZE, layout="constrained")
    for row_index, dataset in enumerate(DATASET_ORDER):
        frame = load_umap(method, dataset)
        ari = adjusted_rand_score(frame.author_cell_type, frame.leiden_cluster)
        categorical_scatter(axes[row_index, 0], frame, "leiden_cluster",
                            f"{DATASET_LABEL[dataset]} — Leiden (ARI={ari:.3f}, {frame.leiden_cluster.nunique()} clusters)", COMPARATOR_POINT_SIZE)
        categorical_scatter(axes[row_index, 1], frame, "author_cell_type",
                            f"{DATASET_LABEL[dataset]} — author cell types ({frame.author_cell_type.nunique()} types)", COMPARATOR_POINT_SIZE, AUTHOR_COLORS)
    fig.suptitle(f"Figure 2 — {METHOD_LABEL[method]} frozen UMAPs", fontsize=PAGE_TITLE_SIZE_PT, fontweight="bold")
    return fig


def make_scknife_page():
    fig, axes = plt.subplots(2, 2, figsize=A4_LANDSCAPE_FIGSIZE, layout="constrained")
    for ax, dataset in zip(axes.flat, SCKNIFE_DATASET_ORDER):
        frame = load_umap("scknife", dataset)
        categorical_scatter(ax, frame, "leiden_cluster",
                            f"{DATASET_LABEL[dataset]} — Leiden ({frame.leiden_cluster.nunique()} clusters)", SCKNIFE_POINT_SIZE)
    fig.suptitle("Figure 2 — scKNIFE frozen UMAPs", fontsize=PAGE_TITLE_SIZE_PT, fontweight="bold")
    return fig

for method in ("expimap", "slalom", "spectra"):
    fig = make_comparator_page(method); display(fig); plt.close(fig)
fig = make_scknife_page(); display(fig); plt.close(fig)

## Tahoe Hallmark panels

Compact local Tahoe inputs transferred from Figure 3.


In [ ]:
# Tahoe Hallmark plots — local compact numerical inputs only.
def tahoe_density_scatter(ax, x, y, cmap, point_size=HALLMARK_DIAGNOSTIC_POINT_SIZE, alpha=HALLMARK_DIAGNOSTIC_ALPHA):
    mask=np.isfinite(x)&np.isfinite(y); x,y=np.asarray(x)[mask],np.asarray(y)[mask]
    hist,xe,ye=np.histogram2d(x,y,bins=120); xi=np.clip(np.searchsorted(xe,x,side="right")-1,0,118); yi=np.clip(np.searchsorted(ye,y,side="right")-1,0,118); density=hist[xi,yi]; order=density.argsort()
    ax.scatter(x[order],y[order],c=density[order],cmap=cmap,s=point_size,alpha=alpha,edgecolors="none",rasterized=True); return x,y
def plot_tahoe_hallmark(driver_gene):
    group=tahoe[driver_gene]; x=group["expression"][:]; scores=[group["bgnmf_score"][:],group["scanpy_score"][:]]
    fig,axes=plt.subplots(1,2,figsize=(14,5.5),dpi=150)
    for ax,y,cmap,color,ylabel,title in zip(axes,scores,[HALLMARK_BGNMF_CMAP,HALLMARK_SCANPY_CMAP],[HALLMARK_BGNMF_DARK,HALLMARK_SCANPY_DARK],["bGNMF Activity Score","Scanpy score_genes"],["bGNMF Factor Score","Scanpy Standard Score"]):
        xp,yp=tahoe_density_scatter(ax,x,y,cmap); slope,intercept=np.polyfit(xp,yp,1); line=np.linspace(xp.min(),xp.max(),200); ax.plot(line,slope*line+intercept,color=color,linewidth=2,linestyle="--")
        ax.set(xlabel=f"{driver_gene} Expression (MAGIC)",ylabel=f"{ylabel} ({group.attrs['pathway_label']})",title=f"{title}\nPearson r = {pearsonr(xp,yp).statistic:.3f} | Spearman rho = {spearmanr(xp,yp).statistic:.3f}"); clean_axis(ax)
    fig.suptitle(f"Hallmark: {group.attrs['pathway_label']} | {group.attrs['drug']}-treated cells | Driver gene: {driver_gene}",fontweight="bold",y=1.02); fig.tight_layout(); plt.show(); return {"driver_gene":driver_gene,"n":len(x)}


### Tahoe Hallmark: TP53


In [ ]:
tahoe_panel_tp53 = plot_tahoe_hallmark("TP53")


### Tahoe Hallmark: PIK3R1


In [ ]:
tahoe_panel_pik3r1 = plot_tahoe_hallmark("PIK3R1")


# Final A4 vector compilation and export

Displays Figure 2 inline and exports the exact A4 canvas as PDF and SVG. Dense UMAP point layers remain rasterized at 300 DPI inside the vector containers.

In [ ]:
# Final A4 portrait mixed raster/vector compilation and export.
# Dense UMAP/Hallmark point clouds are temporary 300-DPI PNG assets;
# axes, labels, legends, annotations, lines, bars, and heatmap cells remain vector.
from pathlib import Path
import json, platform, tempfile
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.image import imread
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter
from scipy.stats import pearsonr, spearmanr
from IPython.display import display
import numpy as np

# ─── Figure 2 editable parameters ───────────────────────────────────────────
PAGE_W_MM, PAGE_H_MM, OUTPUT_DPI = 210, 297, 300
FIG2_FONT_FAMILY = 'DejaVu Sans'
FIG2_FONT_FALLBACK = 'DejaVu Sans'
R_LABEL, R_TITLE, R_TICK, R_NOTE = 7, 7, 7, 7
R_POINT = .055
R_HALLMARK_POINT, R_HALLMARK_ALPHA = 4.5, .85
R_UMAP_LEGEND = 7
REF_B_COLORS = ['#CABCA2', '#C5AD7D', '#BEA890', '#416879', '#416879']
REF_C_COLORS = {'scKNIFE':'#4F8F3A', 'SPECTRA':'#F4C678', 'ExpiMap':'#A31F2A', 'Slalom':'#BFDC80'}

mpl.rcParams.update({
    'font.family': [FIG2_FONT_FAMILY, FIG2_FONT_FALLBACK],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'pdf.use14corefonts': False,
})


def mm_rect(x, y, w, h):
    return [x / PAGE_W_MM, y / PAGE_H_MM, w / PAGE_W_MM, h / PAGE_H_MM]


def panel_title(ax, label, title):
    ax.text(0, 1.035, f'{label}  {title}', transform=ax.transAxes, ha='left', va='bottom',
            fontsize=R_LABEL, fontweight='bold', clip_on=False)


def _square_limits(frame, pad=.04):
    x = np.asarray(frame['umap_1'], dtype=float)
    y = np.asarray(frame['umap_2'], dtype=float)
    x_span, y_span = np.ptp(x), np.ptp(y)
    span = max(x_span, y_span)
    if not np.isfinite(span) or span <= 0:
        span = 1.0
    span *= 1 + pad
    x_mid, y_mid = np.mean(x), np.mean(y)
    return (x_mid - span / 2, x_mid + span / 2), (y_mid - span / 2, y_mid + span / 2)


def _hallmark_limits(x, y, pad=.04):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    x_span = max(np.ptp(x), 1e-9); y_span = max(np.ptp(y), 1e-9)
    return (x.min() - pad * x_span, x.max() + pad * x_span), (y.min() - pad * y_span, y.max() + pad * y_span)


def save_umap_asset(frame, column, colors, path, xlim, ylim, width_mm, height_mm):
    image_fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4), dpi=OUTPUT_DPI, facecolor='white')
    image_ax = image_fig.add_axes([0, 0, 1, 1])
    cats = sorted(frame[column].astype(str).unique(), key=natural_key)
    palette = colors or {value: CATEGORY_PALETTE[i % len(CATEGORY_PALETTE)] for i, value in enumerate(cats)}
    for value in cats:
        take = frame[column].astype(str).eq(value)
        image_ax.scatter(frame.loc[take, 'umap_1'], frame.loc[take, 'umap_2'], s=R_POINT,
                          color=palette[value], alpha=UMAP_ALPHA, linewidths=0)
    image_ax.set_xlim(xlim); image_ax.set_ylim(ylim); image_ax.set_aspect('equal', adjustable='box'); image_ax.set_axis_off()
    image_fig.savefig(path, format='png', dpi=OUTPUT_DPI, facecolor='white', edgecolor='white', pad_inches=0)
    plt.close(image_fig)


def place_umap_asset(ax, path, xlim, ylim, title):
    ax.imshow(imread(path), extent=[xlim[0], xlim[1], ylim[0], ylim[1]], origin='upper',
              aspect='auto', interpolation='none')
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal', adjustable='box')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=R_TITLE, pad=1)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def runtime_celltype_legend(ax, frame, colors):
    ax.axis('off')
    cats = sorted(frame['author_cell_type'].astype(str).unique(), key=natural_key)
    handles = [Line2D([], [], marker='o', linestyle='', markersize=2.8,
                      markerfacecolor=colors[value], markeredgecolor='none', label=value)
               for value in cats]
    ax.legend(handles=handles, loc='upper left', ncol=1, frameon=False,
              fontsize=R_UMAP_LEGEND, handlelength=.45, handletextpad=.20,
              borderaxespad=0, labelspacing=.10, columnspacing=.20, markerscale=1.0)


def draw_runtime_workflow(ax):
    ax.axis('off'); panel_title(ax, 'A', 'Benchmarking workflow')
    stages = [('Cohorts', 'Bassez C1/C2\nWu TNBC · Shiao', '#EAF2F8'),
              ('Methods', 'scKNIFE · SPECTRA\nExpiMap · Slalom', '#F4ECF7'),
              ('Frozen results', 'UMAP · Leiden\nauthor labels', '#E8F6F3'),
              ('Evaluation', 'Runtime · capacity\nARI · AMI · NMI', '#FEF5E7')]
    xs = [.01, .26, .51, .76]
    for i, (heading, detail, color) in enumerate(stages):
        ax.add_patch(FancyBboxPatch((xs[i], .24), .21, .50, boxstyle='round,pad=.008,rounding_size=.015',
                                    linewidth=.7, edgecolor='#555', facecolor=color))
        ax.text(xs[i] + .105, .55, heading, ha='center', va='center', fontsize=R_TITLE, fontweight='bold')
        ax.text(xs[i] + .105, .38, detail, ha='center', va='center', fontsize=R_NOTE, linespacing=1.25)
        if i < 3:
            ax.add_patch(FancyArrowPatch((xs[i] + .215, .49), (xs[i + 1] - .006, .49), arrowstyle='-|>',
                                         mutation_scale=7, linewidth=.7, color='#666'))
    ax.text(.01, .08, 'Shared-cell comparison on frozen coordinates and labels', fontsize=R_NOTE, color='#555')


def draw_runtime_bars(ax):
    datasets = ['Bassez c1', 'Bassez c2']; x = np.arange(2); width = .18
    for i, method in enumerate(METHOD_ORDER):
        values = runtime_df[runtime_df.method.eq(method)].set_index('dataset').reindex(datasets).hours.to_numpy()
        bars = ax.bar(x + (i - 1.5) * width, values, width * .9, color=REF_C_COLORS[method],
                      edgecolor='#444', linewidth=.3, label=method)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + .25, f'{value:.1f}', ha='center', fontsize=R_NOTE, rotation=90)
    ax.set_xticks(x, ['Bassez C1', 'Bassez C2'], fontsize=R_TICK); ax.tick_params(axis='y', labelsize=R_TICK, pad=1)
    ax.set_ylabel('Hours', fontsize=R_TICK, labelpad=1); ax.set_ylim(0, runtime_df.hours.max() * 1.25); panel_title(ax, 'C', 'Training runtime')
    ax.legend(loc='upper left', bbox_to_anchor=(.01, .98), ncol=2, frameon=False, fontsize=R_NOTE,
              columnspacing=.5, handlelength=.8, borderaxespad=0)
    clean_axis(ax)


def _broken_mark(ax, position):
    size = .018; kwargs = {'transform': ax.transAxes, 'color': '#111111', 'clip_on': False, 'linewidth': .65}
    if position == 'bottom':
        ax.plot((-size, +size), (-size, +size), **kwargs); ax.plot((1 - size, 1 + size), (-size, +size), **kwargs)
    else:
        ax.plot((-size, +size), (1 - size, 1 + size), **kwargs); ax.plot((1 - size, 1 + size), (1 - size, 1 + size), **kwargs)


def draw_runtime_capacity_three_tier(fig, x, y, width, height):
    lower = fig.add_axes(mm_rect(x, y, width, height * .30))
    middle = fig.add_axes(mm_rect(x, y + height * .34, width, height * .28), sharex=lower)
    upper = fig.add_axes(mm_rect(x, y + height * .66, width, height * .34), sharex=lower)
    values = capacity_df['count'].to_numpy(); labels = ['SP', 'Ex', 'Sl', 'scK\nbench', 'scK\nfull']; colors = REF_B_COLORS
    for ax in (lower, middle, upper):
        ax.bar(range(5), values, color=colors, edgecolor='#3A2D29', linewidth=.55, width=.64)
        ax.set_xlim(-.6, 4.6); clean_axis(ax); ax.tick_params(axis='y', labelsize=R_TICK, pad=1)
    lower.set_ylim(0, 220); middle.set_ylim(700, 1100); upper.set_ylim(20000, 42000)
    comma_fmt = FuncFormatter(lambda value, _: f'{int(value):,}')
    for ax, ticks in ((lower, [0, 100, 200]), (middle, [700, 850, 1000]), (upper, [20000, 30000, 40000])):
        ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_major_formatter(comma_fmt)
    lower.set_xticks(range(5), labels, fontsize=R_TICK)
    for ax in (middle, upper): ax.tick_params(axis='x', bottom=False, labelbottom=False)
    lower.set_ylabel('Features', fontsize=R_TICK, labelpad=1); panel_title(upper, 'B', 'Prior knowledge')
    def label_visible(ax, ymin, ymax):
        offset = (ymax - ymin) * .045
        for index, value in enumerate(values):
            if ymin <= value <= ymax:
                ax.text(index, value + offset, f'{value:,}', ha='center', va='bottom', fontsize=R_NOTE,
                        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': .90, 'pad': .55})
    label_visible(lower, 0, 220); label_visible(middle, 700, 1100); label_visible(upper, 20000, 42000)
    for ax in (lower, middle, upper):
        ax.spines['top'].set_visible(False); ax.spines['bottom'].set_visible(False)
    lower.spines['bottom'].set_visible(True)
    _broken_mark(lower, 'top'); _broken_mark(middle, 'bottom'); _broken_mark(middle, 'top'); _broken_mark(upper, 'bottom')


def runtime_metrics(ax):
    rows = [(d, m) for d in DATASET_ORDER for m in METHOD_ORDER]
    values = np.array([[agreement_df.loc[agreement_df.dataset.eq(d) & agreement_df.method.eq(m), v].iloc[0]
                        for v in ('ari', 'nmi', 'ami')] for d, m in rows])
    cmap = mpl.colormaps[METRIC_CMAP]
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            ax.add_patch(Rectangle((col - .5, row - .5), 1, 1, facecolor=cmap(values[row, col]), edgecolor='none'))
    ax.set_xlim(-.5, values.shape[1] - .5); ax.set_ylim(values.shape[0] - .5, -.5)
    ax.set_xticks(range(3), ['ARI', 'NMI', 'AMI'], fontsize=R_TICK)
    ax.set_yticks(range(12), [m for d, m in rows], fontsize=R_TICK); ax.tick_params(axis='y', pad=1)
    for boundary in (3.5, 7.5): ax.axhline(boundary, color='white', linewidth=1.2)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]; rgba = cmap(value); lum = .2126 * rgba[0] + .7152 * rgba[1] + .0722 * rgba[2]
            ax.text(col, row, f'{value:.2f}', ha='center', va='center', fontsize=R_NOTE, color='black' if lum > .58 else 'white')
    panel_title(ax, 'E', 'Agreement metrics')
    ax.text(0, 1.01, 'row blocks: Bassez C1 · Bassez C2 · Wu TNBC', transform=ax.transAxes, fontsize=R_NOTE, ha='left', va='bottom')
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_color('#000000'); spine.set_linewidth(.55)
    clean_axis(ax)
    return cmap, 0.0, 1.0


def vector_colorbar(fig, rect, cmap, vmin, vmax, label=''):
    cax = fig.add_axes(rect); edges = np.linspace(vmin, vmax, 129)
    for start, end in zip(edges[:-1], edges[1:]):
        cax.add_patch(Rectangle((0, start), 1, end - start, facecolor=cmap((start - vmin) / (vmax - vmin)), edgecolor='none'))
    cax.set_xlim(0, 1); cax.set_ylim(vmin, vmax); cax.set_xticks([]); cax.set_yticks(np.linspace(vmin, vmax, 5))
    cax.tick_params(labelsize=R_TICK, length=1, pad=1, width=.3)
    for spine in cax.spines.values():
        spine.set_visible(True); spine.set_linewidth(.25); spine.set_color('black')
    if label: cax.set_ylabel(label, fontsize=R_TICK, labelpad=1)


def save_hallmark_asset(expression, score, cmap, path, xlim, ylim, width_mm, height_mm):
    image_fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4), dpi=OUTPUT_DPI, facecolor='white')
    image_ax = image_fig.add_axes([0, 0, 1, 1])
    xp, yp = tahoe_density_scatter(image_ax, expression, score, cmap, point_size=R_HALLMARK_POINT, alpha=R_HALLMARK_ALPHA)
    image_ax.set_xlim(xlim); image_ax.set_ylim(ylim); image_ax.set_axis_off()
    image_fig.savefig(path, format='png', dpi=OUTPUT_DPI, facecolor='white', edgecolor='white', pad_inches=0)
    plt.close(image_fig)
    return xp, yp


def draw_tahoe_compiled(fig, x, y, width, height, gene, label, asset_dir, asset_records):
    group = tahoe[gene]; expression = group['expression'][:]; drug = str(group.attrs['drug'])
    drug_label = 'Doxo' if 'Doxorubicin' in drug else ('Afi' if 'Afatinib' in drug else drug)
    for index, (score, cmap, line_color, title) in enumerate(((group['bgnmf_score'][:], HALLMARK_BGNMF_CMAP, HALLMARK_BGNMF_DARK, 'scKNIFE'),
                                                               (group['scanpy_score'][:], HALLMARK_SCANPY_CMAP, HALLMARK_SCANPY_DARK, 'Scanpy'))):
        xlim, ylim = _hallmark_limits(expression, score)
        asset_path = asset_dir / f'Figure_2_{gene}_{title}_Hallmark_points.png'
        xp, yp = save_hallmark_asset(expression, score, cmap, asset_path, xlim, ylim, width / 2 - 2, height)
        asset_records.append({'filename': asset_path.name, 'role': 'Hallmark point cloud', 'dpi': OUTPUT_DPI})
        ax = fig.add_axes(mm_rect(x + index * (width / 2 + 2), y, width / 2 - 2, height))
        ax.imshow(imread(asset_path), extent=[xlim[0], xlim[1], ylim[0], ylim[1]], origin='upper', aspect='auto', interpolation='none')
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        slope, intercept = np.polyfit(xp, yp, 1); line = np.linspace(xp.min(), xp.max(), 100)
        ax.plot(line, slope * line + intercept, color=line_color, linestyle='--', linewidth=.6)
        pearson_value = pearsonr(xp, yp).statistic; spearman_value = spearmanr(xp, yp).statistic
        ax.text(.03, .97, f'Pearson r = {pearson_value:.2f}\nSpearman ρ = {spearman_value:.2f}', transform=ax.transAxes,
                ha='left', va='top', fontsize=R_NOTE, color='#333333', bbox={'facecolor':'white', 'edgecolor':'none', 'alpha':.78, 'pad':.7})
        ax.set_title(f'{label} · {drug_label} | {title}', fontsize=R_TITLE, pad=1)
        ax.set_xlabel(gene, fontsize=R_TICK, labelpad=1); ax.set_ylabel('Score', fontsize=R_TICK, labelpad=1)
        ax.tick_params(labelsize=R_NOTE, pad=1); clean_axis(ax)


def make_figure2_runtime_preview():
    fig = plt.figure(figsize=(PAGE_W_MM / 25.4, PAGE_H_MM / 25.4), facecolor='white')
    asset_records = []
    with tempfile.TemporaryDirectory(prefix='figure_2_raster_') as temp_dir:
        asset_dir = Path(temp_dir)
        draw_runtime_workflow(fig.add_axes(mm_rect(7, 238, 91, 52)))
        draw_runtime_capacity_three_tier(fig, 108, 238, 34, 52)
        draw_runtime_bars(fig.add_axes(mm_rect(150, 238, 53, 52)))
        umap_x, umap_w, umap_gap = 7, 125, 3; umap_each = (umap_w - 3 * umap_gap) / 4
        cluster_y, cluster_h = 190, 34; celltype_y, celltype_h = 147, 34; legend_y, legend_h = 84, 54
        for i, dataset in enumerate(SCKNIFE_DATASET_ORDER):
            x = umap_x + i * (umap_each + umap_gap); frame = load_umap('scknife', dataset); xlim, ylim = _square_limits(frame)
            cluster_path = asset_dir / f'Figure_2_{dataset}_clustering_UMAP.png'
            save_umap_asset(frame, 'leiden_cluster', None, cluster_path, xlim, ylim, umap_each, cluster_h)
            asset_records.append({'filename': cluster_path.name, 'role': 'UMAP point cloud', 'dpi': OUTPUT_DPI})
            cluster_ax = fig.add_axes(mm_rect(x, cluster_y, umap_each, cluster_h))
            place_umap_asset(cluster_ax, cluster_path, xlim, ylim, DATASET_LABEL[dataset])
            if i == 0: panel_title(cluster_ax, 'D', 'Clustering')
            celltype_path = asset_dir / f'Figure_2_{dataset}_cell_type_UMAP.png'
            save_umap_asset(frame, 'author_cell_type', AUTHOR_COLORS, celltype_path, xlim, ylim, umap_each, celltype_h)
            asset_records.append({'filename': celltype_path.name, 'role': 'UMAP point cloud', 'dpi': OUTPUT_DPI})
            celltype_ax = fig.add_axes(mm_rect(x, celltype_y, umap_each, celltype_h))
            place_umap_asset(celltype_ax, celltype_path, xlim, ylim, DATASET_LABEL[dataset])
            if i == 0: panel_title(celltype_ax, 'F', 'Cell type')
            runtime_celltype_legend(fig.add_axes(mm_rect(x, legend_y, umap_each, legend_h)), frame, AUTHOR_COLORS)
        metric_ax = fig.add_axes(mm_rect(163, 135, 37, 79)); metric_cmap, metric_vmin, metric_vmax = runtime_metrics(metric_ax)
        vector_colorbar(fig, mm_rect(202, 141, 1.5, 65), metric_cmap, metric_vmin, metric_vmax)
        draw_tahoe_compiled(fig, 19, 50, 80, 34, 'TP53', 'G  TP53', asset_dir, asset_records)
        draw_tahoe_compiled(fig, 111, 50, 80, 34, 'PIK3R1', 'H  PIK3R1', asset_dir, asset_records)
    return fig, asset_records




FIGURE2_PANEL_SIZES_MM = {
    "A": (96, 57), "B": (38, 57), "C": (57, 57), "D": (130, 39),
    "E": (62, 86), "F": (130, 103), "G": (84, 40), "H": (84, 40),
}


def make_figure2_panel(label):
    # Plot one authoritative Figure 2 panel directly from validated data.
    global PAGE_W_MM, PAGE_H_MM
    if label not in FIGURE2_PANEL_SIZES_MM:
        raise ValueError(f"Unknown Figure 2 panel: {label}")
    previous_size = (PAGE_W_MM, PAGE_H_MM)
    PAGE_W_MM, PAGE_H_MM = FIGURE2_PANEL_SIZES_MM[label]
    width, height = PAGE_W_MM, PAGE_H_MM
    fig = plt.figure(
        figsize=(width / 25.4, height / 25.4),
        facecolor="white",
    )
    asset_records = []
    try:
        with tempfile.TemporaryDirectory(prefix=f"figure_2_panel_{label}_") as temp_dir:
            asset_dir = Path(temp_dir)
            if label == "A":
                draw_runtime_workflow(fig.add_axes(mm_rect(2, 3, width - 4, height - 6)))
            elif label == "B":
                draw_runtime_capacity_three_tier(fig, 2, 3, width - 4, height - 6)
            elif label == "C":
                draw_runtime_bars(fig.add_axes(mm_rect(3, 4, width - 6, height - 8)))
            elif label in {"D", "F"}:
                margin, gap = 2.0, 2.0
                each = (width - 2 * margin - 3 * gap) / 4
                for index, dataset in enumerate(SCKNIFE_DATASET_ORDER):
                    x = margin + index * (each + gap)
                    frame = load_umap("scknife", dataset)
                    xlim, ylim = _square_limits(frame)
                    if label == "D":
                        image_path = asset_dir / f"{dataset}_clustering.png"
                        save_umap_asset(frame, "leiden_cluster", None, image_path, xlim, ylim, each, height - 6)
                        axis = fig.add_axes(mm_rect(x, 3, each, height - 6))
                        place_umap_asset(axis, image_path, xlim, ylim, DATASET_LABEL[dataset])
                        if index == 0:
                            panel_title(axis, "D", "Clustering")
                    else:
                        plot_height = height * .43
                        image_path = asset_dir / f"{dataset}_cell_type.png"
                        save_umap_asset(frame, "author_cell_type", AUTHOR_COLORS, image_path, xlim, ylim, each, plot_height)
                        axis = fig.add_axes(mm_rect(x, height - plot_height - 3, each, plot_height))
                        place_umap_asset(axis, image_path, xlim, ylim, DATASET_LABEL[dataset])
                        if index == 0:
                            panel_title(axis, "F", "Cell type")
                        runtime_celltype_legend(fig.add_axes(mm_rect(x, 2, each, height - plot_height - 7)), frame, AUTHOR_COLORS)
            elif label == "E":
                metric_grid = fig.add_gridspec(
                    1, 2, width_ratios=(1, 0.055), wspace=0.12,
                    left=0.28, right=0.88, bottom=0.07, top=0.92,
                )
                metric_axis = fig.add_subplot(metric_grid[0, 0])
                metric_cmap, metric_vmin, metric_vmax = runtime_metrics(metric_axis)
                metric_colorbar_axis = fig.add_subplot(metric_grid[0, 1])
                metric_colorbar = fig.colorbar(
                    mpl.cm.ScalarMappable(
                        norm=mpl.colors.Normalize(metric_vmin, metric_vmax), cmap=metric_cmap,
                    ),
                    cax=metric_colorbar_axis,
                    ticks=np.linspace(metric_vmin, metric_vmax, 5),
                )
                metric_colorbar.ax.yaxis.set_ticks_position("right")
                metric_colorbar.ax.tick_params(labelsize=R_TICK, length=1, pad=1, width=.3)
            elif label in {"G", "H"}:
                gene = {"G": "TP53", "H": "PIK3R1"}[label]
                lineage = _data_manifest["panels"][label]
                expected_selectors = {gene, f"{gene}/expression", f"{gene}/bgnmf_score", f"{gene}/scanpy_score"}
                declared_selectors = set(lineage["inputs"][0].get("selectors", []))
                if lineage.get("title") != gene or declared_selectors != expected_selectors:
                    raise RuntimeError(f"Panel {label} lineage does not match plotted gene {gene}")
                draw_tahoe_compiled(fig, 3, 3, width - 6, height - 6, gene, f"{label}  {gene}", asset_dir, asset_records)
        return fig
    finally:
        PAGE_W_MM, PAGE_H_MM = previous_size


In [ ]:

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


def _save_live_vector_svg(figure, path, bbox_inches, visible_axes, visible_texts, pad_inches=0):
    """Save editable SVG text/geometry while keeping dense AxesImage layers rasterized."""
    axis_states = [(axis, axis.get_visible()) for axis in figure.axes]
    text_states = [(item, item.get_visible()) for item in figure.texts]
    visible_axis_ids = {id(axis) for axis in visible_axes}
    visible_text_ids = {id(item) for item in visible_texts}
    try:
        for axis, _ in axis_states:
            axis.set_visible(id(axis) in visible_axis_ids)
        for item, _ in text_states:
            item.set_visible(id(item) in visible_text_ids)
        with mpl.rc_context({"svg.fonttype": "none", "svg.image_inline": True}):
            figure.savefig(path, format="svg", facecolor="white", bbox_inches=bbox_inches,
                           pad_inches=pad_inches, metadata={"Creator": "Matplotlib mixed raster/vector renderer"})
    finally:
        for axis, state in axis_states:
            axis.set_visible(state)
        for item, state in text_states:
            item.set_visible(state)


# Shared export setup. Every panel cell below plots one labeled panel directly
# from validated intermediate data, saves it, and displays it with plt.show().
import json
from matplotlib.transforms import Bbox
_compiled_dir = Path('figs/panel/Figure_2').resolve()
_individual_dir = Path('figs/individual/Figure_2').resolve()
_compiled_dir.mkdir(parents=True, exist_ok=True); _individual_dir.mkdir(parents=True, exist_ok=True)
(_individual_dir / 'Figure_2_panel_crop_manifest.json').unlink(missing_ok=True)
_data_manifest_path = Path('data/Figure_2/manifest.json')
_data_manifest = json.loads(_data_manifest_path.read_text())
if _data_manifest.get('format_version') != 2:
    raise RuntimeError(f'{_data_manifest_path} must use manifest format_version 2')
if set(_data_manifest.get('panels', {})) != set('ABCDEFGH'):
    raise RuntimeError(f'{_data_manifest_path} panel declarations do not match ABCDEFGH')
_panel_manifest = {}
_standalone_export_pad_inches = 0.04
print('Data-first standalone panel export helpers ready with validated panel lineage.')


## Figure 2A — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['A']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2A has no declared intermediate inputs')
_panel_figure = make_figure2_panel('A')

_png = _individual_dir / 'Figure_2_A.png'
_pdf = _individual_dir / 'Figure_2_A.pdf'
_svg = _individual_dir / 'Figure_2_A.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['A'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2B — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['B']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2B has no declared intermediate inputs')
_panel_figure = make_figure2_panel('B')

_png = _individual_dir / 'Figure_2_B.png'
_pdf = _individual_dir / 'Figure_2_B.pdf'
_svg = _individual_dir / 'Figure_2_B.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['B'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2C — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['C']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2C has no declared intermediate inputs')
_panel_figure = make_figure2_panel('C')

_png = _individual_dir / 'Figure_2_C.png'
_pdf = _individual_dir / 'Figure_2_C.pdf'
_svg = _individual_dir / 'Figure_2_C.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['C'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2D — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['D']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2D has no declared intermediate inputs')
_panel_figure = make_figure2_panel('D')

_png = _individual_dir / 'Figure_2_D.png'
_pdf = _individual_dir / 'Figure_2_D.pdf'
_svg = _individual_dir / 'Figure_2_D.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['D'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2E — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['E']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2E has no declared intermediate inputs')
_panel_figure = make_figure2_panel('E')

_png = _individual_dir / 'Figure_2_E.png'
_pdf = _individual_dir / 'Figure_2_E.pdf'
_svg = _individual_dir / 'Figure_2_E.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['E'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2F — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['F']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2F has no declared intermediate inputs')
_panel_figure = make_figure2_panel('F')

_png = _individual_dir / 'Figure_2_F.png'
_pdf = _individual_dir / 'Figure_2_F.pdf'
_svg = _individual_dir / 'Figure_2_F.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['F'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2G — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['G']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2G has no declared intermediate inputs')
_panel_figure = make_figure2_panel('G')

_png = _individual_dir / 'Figure_2_G.png'
_pdf = _individual_dir / 'Figure_2_G.pdf'
_svg = _individual_dir / 'Figure_2_G.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['G'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Figure 2H — independently plotted panel

This cell plots the labeled panel directly from validated intermediate data, then immediately saves PNG, PDF, and SVG and calls plt.show().

In [ ]:

# Direct plotting call from intermediate data; no compiled canvas or saved image is used.
_panel_lineage = _data_manifest['panels']['H']
if _panel_lineage['kind'] == 'data' and not _panel_lineage['inputs']:
    raise RuntimeError('Data panel Figure_2H has no declared intermediate inputs')
_panel_figure = make_figure2_panel('H')

_png = _individual_dir / 'Figure_2_H.png'
_pdf = _individual_dir / 'Figure_2_H.pdf'
_svg = _individual_dir / 'Figure_2_H.svg'
_panel_figure.savefig(
    _png, format='png', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_panel_figure.savefig(
    _pdf, format='pdf', dpi=300, facecolor='white',
    bbox_inches='tight', pad_inches=_standalone_export_pad_inches,
)
_save_live_vector_svg(
    _panel_figure, _svg, 'tight', list(_panel_figure.axes), list(_panel_figure.texts),
    pad_inches=_standalone_export_pad_inches,
)
_panel_manifest['H'] = {
    'mode': 'standalone_data_driven_figure',
    'renderer': 'matplotlib_fallback',
    'matplotlib_version': mpl.__version__,
    'axis_count': len(_panel_figure.axes),
    'figure_text_count': len(_panel_figure.texts),
    'svg_text_mode': 'editable_text', 'svg_raster_images': sum(len(axis.images) for axis in _panel_figure.axes),
    'saved_in_individual_notebook_cell': True,
    'compiled_canvas_crop': False,
    'fixed_canvas_export': False,
    'export_bounds': 'content_aware_tight',
    'notebook_display': 'matplotlib.pyplot.show',
    'data_lineage_kind': _panel_lineage['kind'],
    'intermediate_inputs': _panel_lineage['inputs'],
    'data_extractor': _panel_lineage['extractor'],
    'extraction_mode': _panel_lineage['extraction_mode'],
    'outputs': {path.suffix.lstrip('.'): {'path': path.relative_to(Path.cwd()).as_posix(), 'sha256': sha256(path)} for path in (_png, _pdf, _svg)},
}
(_individual_dir / 'Figure_2_panel_export_manifest.json').write_text(json.dumps(_panel_manifest, indent=2) + '\n')
plt.figure(_panel_figure.number)
plt.show()
print('Live Jupyter plot saved:', _png.name, _pdf.name, _svg.name)
plt.close(_panel_figure)


## Compiled figure — assembled after individual plots

In [ ]:
# Build the compiled figure only after every independently plotted panel above.
_submission_figure, _submission_assets = make_figure2_runtime_preview()
_compiled_outputs = {}
for _suffix in ("png", "pdf"):
    _path = _compiled_dir / f"Figure_2_main_panel.{_suffix}"
    _submission_figure.savefig(_path, format=_suffix, dpi=OUTPUT_DPI, facecolor="white")
    _compiled_outputs[_suffix] = {"path": _path.relative_to(Path.cwd()).as_posix(), "sha256": sha256(_path)}
_svg_path = _compiled_dir / "Figure_2_main_panel.svg"
_full_bbox = Bbox.from_bounds(0, 0, *_submission_figure.get_size_inches())
_save_live_vector_svg(_submission_figure, _svg_path, _full_bbox, list(_submission_figure.axes), list(_submission_figure.texts))
_compiled_outputs["svg"] = {"path": _svg_path.relative_to(Path.cwd()).as_posix(), "sha256": sha256(_svg_path)}
_compiled_manifest = {
    "canvas_mm": [PAGE_W_MM, PAGE_H_MM], "dpi": OUTPUT_DPI,
    "hallmark_panels": {"G": "TP53", "H": "PIK3R1"},
    "raster_assets": _submission_assets,
    "svg_raster_images": sum(len(axis.images) for axis in _submission_figure.axes),
    "rasterization": "Dense UMAP and Hallmark point clouds are embedded 300-DPI raster layers; text, axes, bars, workflow shapes, metric cells, and annotations remain vector.",
    "text_rendering": "Editable SVG text and embedded TrueType PDF fonts.",
    "renderer": "matplotlib mixed raster/vector fallback", "matplotlib": mpl.__version__, "python": sys.version.split()[0],
    "outputs": _compiled_outputs,
}
_manifest_path = _compiled_dir / "Figure_2_main_panel_manifest.json"
_manifest_path.write_text(json.dumps(_compiled_manifest, indent=2) + "\n")
plt.figure(_submission_figure.number); plt.show()
print("Compiled Figure 2 and current manifest saved as PNG, PDF, and compact SVG.")
plt.close(_submission_figure)


# Supplementary Figure 2 — frozen no-match UMAP panels

This separate A4 supplementary compilation displays the supplied frozen no-match UMAP panels.


In [ ]:
# Supplementary Figure 2 — separate A4 mixed raster/vector compilation and export.
# Uses raster derivatives of the validated frozen Figure 2 coordinates and labels; no clustering or embedding is run here.

from pathlib import Path
import hashlib
import json
import re
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from IPython.display import display

mpl.rcParams.update({"svg.fonttype": "none", "pdf.fonttype": 42, "pdf.use14corefonts": False})

SUPP_PAGE_W_MM, SUPP_PAGE_H_MM = 210, 297
SUPP_LABEL_PT = 7
SUPP_TITLE_PT = 7
SUPP_TICK_PT = 7
SUPP_NOTE_PT = 7
SUPP_LEGEND_PT = 7
SUPP_LEGEND_MARKER_PT = 3.5
SUPP_DATASET_ORDER = DATASET_ORDER.copy()
SUPP_DATA_DIR = DATA_DIR / "supplementary_umap_no_match"
SUPP_MANIFEST_PATH = SUPP_DATA_DIR / "no_match_umap_manifest.json"
SUPP_UMAP_METHODS = [("expimap", "ExpiMap"), ("slalom", "Slalom"), ("spectra", "SPECTRA")]
SUPP_UMAP_FILES = {(slug, dataset): SUPP_DATA_DIR / f"{slug}_{dataset}_points.png" for slug, _ in SUPP_UMAP_METHODS for dataset in SUPP_DATASET_ORDER}
SUPP_W_DATA_DIR = DATA_DIR / "supplementary_w_pooled"
SUPP_CONCORDANCE_PATH = SUPP_W_DATA_DIR / "w_pooled_concordance_metrics.csv"
SUPP_CONCORDANCE_MANIFEST_PATH = SUPP_W_DATA_DIR / "w_pooled_concordance_manifest.json"
SUPP_TSNE_PATHS = [
    SUPP_W_DATA_DIR / "w_pooled_tsne_dataset_panels.png",
    SUPP_W_DATA_DIR / "w_pooled_tsne_node_kinds_corrected.png",
]
SUPP_CONCORDANCE_DISPLAY_W_MM = 98
SUPP_CONCORDANCE_X_MM = 5
SUPP_CONCORDANCE_Y_MM = 16
SUPP_TSNE_DISPLAY_W_MM = 96
SUPP_TSNE_X_MM = 109
SUPP_TSNE_Y_MM = 10
SUPP_TSNE_GAP_MM = 4
if not SUPP_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing no-match UMAP manifest: {SUPP_MANIFEST_PATH}")
if not SUPP_CONCORDANCE_PATH.is_file():
    raise FileNotFoundError(f"Missing pooled concordance data: {SUPP_CONCORDANCE_PATH}")
if not SUPP_CONCORDANCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing pooled concordance manifest: {SUPP_CONCORDANCE_MANIFEST_PATH}")
for tsne_path in SUPP_TSNE_PATHS:
    if not tsne_path.is_file():
        raise FileNotFoundError(f"Missing pooled W-analysis t-SNE panel: {tsne_path}")
SUPP_MANIFEST = json.loads(SUPP_MANIFEST_PATH.read_text())
if SUPP_MANIFEST.get("schema_version") != "no_match_umap_raster_v3":
    raise RuntimeError(f"Unexpected no-match manifest schema: {SUPP_MANIFEST.get('schema_version')}")
if SUPP_MANIFEST.get("source_mode") != "validated_frozen_figure2_bundle":
    raise RuntimeError("Supplementary Figure 2 requires derivatives of the validated frozen Figure 2 bundle")
expected_output_keys = {f"{slug}/{dataset}" for slug, _ in SUPP_UMAP_METHODS for dataset in SUPP_DATASET_ORDER}
if set(SUPP_MANIFEST.get("outputs", {})) != expected_output_keys:
    raise RuntimeError("No-match manifest does not contain exactly the nine expected method/cohort panels")

def supp_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for slug, _ in SUPP_UMAP_METHODS:
    for dataset in SUPP_DATASET_ORDER:
        key = f"{slug}/{dataset}"
        entry = SUPP_MANIFEST["outputs"][key]
        raster_path = SUPP_UMAP_FILES[(slug, dataset)]
        table_path = DATA_DIR / f"{slug}_{dataset}_umap.csv.gz"
        if not raster_path.is_file():
            raise FileNotFoundError(f"Missing packaged frozen raster: {raster_path}")
        if supp_sha256(raster_path) != entry["combined_png_sha256"]:
            raise RuntimeError(f"Manifest checksum mismatch for {raster_path}")
        if supp_sha256(table_path) != entry["source_sha256"]:
            raise RuntimeError(f"Frozen source-table checksum mismatch for {table_path}")
        frame = pd.read_csv(table_path, usecols=["leiden_cluster", "author_cell_type"], dtype=str)
        if len(frame) != entry["cell_count"]:
            raise RuntimeError(f"Frozen cell count mismatch for {key}")
        if set(frame["leiden_cluster"]) != set(entry["leiden_colors"]):
            raise RuntimeError(f"Frozen Leiden labels mismatch for {key}")
        if set(frame["author_cell_type"]) != set(entry["author_colors"]):
            raise RuntimeError(f"Frozen author labels mismatch for {key}")

def supp_natural_key(value):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", str(value))]

def supp_rect(x_mm, y_mm, width_mm, height_mm):
    return [
        x_mm / SUPP_PAGE_W_MM,
        y_mm / SUPP_PAGE_H_MM,
        width_mm / SUPP_PAGE_W_MM,
        height_mm / SUPP_PAGE_H_MM,
    ]

def supp_frozen_legend_specs(slug, dataset):
    key = f"{slug}/{dataset}"
    entry = SUPP_MANIFEST["outputs"][key]
    leiden_order = sorted(entry["leiden_colors"], key=lambda value: int(value))
    author_order = sorted(entry["author_colors"], key=supp_natural_key)
    cluster_handles = [
        Line2D(
            [], [], marker="o", linestyle="", markersize=SUPP_LEGEND_MARKER_PT,
            markerfacecolor=entry["leiden_colors"][label],
            markeredgecolor="none", label=label,
        )
        for label in leiden_order
    ]
    author_handles = [
        Line2D(
            [], [], marker="o", linestyle="", markersize=SUPP_LEGEND_MARKER_PT,
            markerfacecolor=entry["author_colors"][label],
            markeredgecolor="none", label=label,
        )
        for label in author_order
    ]
    return cluster_handles, author_handles

def supp_overlay_vector_content(fig, title_rect, legend_rect, slug, dataset):
    cluster_handles, author_handles = supp_frozen_legend_specs(slug, dataset)
    title_axis = fig.add_axes(title_rect, frameon=False, zorder=10)
    title_axis.set_axis_off()
    title_axis.text(0.25, 0.50, "Leiden", ha="center", va="center", fontsize=SUPP_TITLE_PT)
    title_axis.text(0.75, 0.50, "Author cell type", ha="center", va="center", fontsize=SUPP_TITLE_PT)

    legend_axis = fig.add_axes(legend_rect, frameon=False, zorder=10)
    legend_axis.set_axis_off()
    legend_common = {
        "frameon": False,
        "fontsize": SUPP_LEGEND_PT,
        "handlelength": 0.55,
        "handletextpad": 0.20,
        "borderaxespad": 0,
        "borderpad": 0,
        "labelspacing": 0.08,
        "columnspacing": 0.45,
        "markerscale": 1.0,
    }
    cluster_legend = legend_axis.legend(
        handles=cluster_handles,
        loc="upper left",
        bbox_to_anchor=(0.00, 1.00),
        ncol=2 if len(cluster_handles) > 8 else 1,
        **legend_common,
    )
    legend_axis.add_artist(cluster_legend)
    legend_axis.legend(
        handles=author_handles,
        loc="upper left",
        bbox_to_anchor=(0.52, 1.00),
        ncol=1,
        **legend_common,
    )

def draw_supp_nomatch_umaps(fig):
    x0_mm, y_top_mm = 15, 214
    cell_width_mm, cell_height_mm = 58, 40
    plot_height_mm, legend_height_mm, title_height_mm = 18, 18, 4
    col_gap_mm, row_gap_mm = 4, 4
    for col_index, dataset in enumerate(SUPP_DATASET_ORDER):
        center_x = x0_mm + col_index * (cell_width_mm + col_gap_mm) + cell_width_mm / 2
        fig.text(
            center_x / SUPP_PAGE_W_MM,
            259 / SUPP_PAGE_H_MM,
            DATASET_LABEL[dataset],
            ha="center",
            va="bottom",
            fontsize=SUPP_TITLE_PT,
        )
    for row_index, (slug, method) in enumerate(SUPP_UMAP_METHODS):
        y_mm = y_top_mm - row_index * (cell_height_mm + row_gap_mm)
        fig.text(
            11 / SUPP_PAGE_W_MM,
            (y_mm + cell_height_mm / 2) / SUPP_PAGE_H_MM,
            method,
            ha="center",
            va="center",
            rotation=90,
            fontsize=SUPP_TITLE_PT,
        )
        for col_index, dataset in enumerate(SUPP_DATASET_ORDER):
            x_mm = x0_mm + col_index * (cell_width_mm + col_gap_mm)
            image_rect = supp_rect(
                x_mm, y_mm + legend_height_mm, cell_width_mm, plot_height_mm
            )
            title_rect = supp_rect(
                x_mm, y_mm + legend_height_mm + plot_height_mm, cell_width_mm, title_height_mm
            )
            legend_rect = supp_rect(
                x_mm, y_mm, cell_width_mm, legend_height_mm
            )
            ax = fig.add_axes(image_rect, zorder=0)
            ax.imshow(plt.imread(SUPP_UMAP_FILES[(slug, dataset)]), aspect="auto", interpolation="none")
            ax.axis("off")
            supp_overlay_vector_content(fig, title_rect, legend_rect, slug, dataset)


def draw_supp_concordance(fig):
    metrics = pd.read_csv(SUPP_CONCORDANCE_PATH)
    order = ["Bassez_A", "Bassez_B", "Shiao", "Wu"]
    labels = {"Bassez_A":"Bassez C1", "Bassez_B":"Bassez C2", "Shiao":"Shiao", "Wu":"Wu TNBC"}
    matrix = np.eye(len(order), dtype=float)
    for row in metrics.loc[metrics.record_type.eq("pair")].itertuples():
        i, j = order.index(row.dataset_a), order.index(row.dataset_b)
        matrix[i, j] = matrix[j, i] = float(row.linear_cka)
    cmap, norm = plt.get_cmap("viridis"), mpl.colors.Normalize(0, 1)
    fig.text(7/SUPP_PAGE_W_MM, 75/SUPP_PAGE_H_MM, "Cross-dataset W concordance", fontsize=SUPP_TITLE_PT, fontweight="bold")
    ax = fig.add_axes(supp_rect(7, 30, 37, 37))
    for i in range(4):
        for j in range(4):
            value = matrix[i, j]
            ax.add_patch(Rectangle((j-.5, i-.5), 1, 1, facecolor=cmap(norm(value)), edgecolor="white", linewidth=.35))
            ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=5.2, color="white" if value >= .58 else "black")
    ax.set(xlim=(-.5,3.5), ylim=(3.5,-.5), xlabel="Dataset", ylabel="Dataset")
    ax.set_xticks(range(4), [labels[d] for d in order], rotation=35, ha="right", fontsize=5.2)
    ax.set_yticks(range(4), [labels[d] for d in order], fontsize=5.2)
    ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
    cax = fig.add_axes(supp_rect(46, 30, 2, 37))
    color_steps = 100
    for step in range(color_steps):
        cax.add_patch(Rectangle((0, step/color_steps), 1, 1/color_steps, facecolor=cmap((step+.5)/color_steps), edgecolor="none"))
    cax.set(xlim=(0,1), ylim=(0,1), ylabel="Linear CKA")
    cax.set_xticks([]); cax.set_yticks(np.linspace(0,1,6)); cax.tick_params(labelsize=5.2, length=1.5)
    cax.yaxis.set_label_position("right"); cax.yaxis.tick_right(); cax.yaxis.label.set_size(5.2)
    table = fig.add_axes(supp_rect(53, 28, 48, 40), frameon=False); table.set_axis_off()
    table.set_title("Within-dataset factor diagnostics", loc="left", fontsize=SUPP_TITLE_PT, fontweight="bold")
    headers = ("Dataset", "Factors", "Eff. rank", "Med. |r|", "P95 |r|", "Condition")
    xpos = (0, .24, .44, .65, .82, 1.0)
    for x, value in zip(xpos, headers):
        table.text(x, .84, value, ha="left" if x == 0 else "right", fontsize=4.5, fontweight="bold", transform=table.transAxes)
    table.plot((0,1),(.78,.78), color="#555555", linewidth=.5, transform=table.transAxes)
    stats = metrics.loc[metrics.record_type.eq("dataset")].set_index("dataset_a")
    for y, dataset in zip((.64,.49,.34,.19), order):
        row = stats.loc[dataset]
        values = (labels[dataset], f"{int(row.n_factors)}", f"{row.effective_rank:.1f} ({100*row.effective_rank_fraction:.0f}%)", f"{row.median_abs_factor_correlation:.3f}", f"{row.p95_abs_factor_correlation:.3f}", f"{row.condition_number:.2f}")
        for x, value in zip(xpos, values): table.text(x, y, value, ha="left" if x == 0 else "right", fontsize=4.5, transform=table.transAxes)
    table.add_patch(Rectangle((0,.01),1,.10,facecolor="#F1F5F4",edgecolor="#8DA6A0",linewidth=.5,transform=table.transAxes))
    table.text(.5,.06,"Concordant W structure with limited factor redundancy",ha="center",va="center",fontsize=4.5,fontweight="bold",color="#244B45",transform=table.transAxes)
    fig.text(7/SUPP_PAGE_W_MM, 20/SUPP_PAGE_H_MM, "CKA quantifies shared node geometry; effective rank and factor correlations quantify within-run redundancy.", fontsize=4.5, color="#444444")


def draw_supp_tsne_panels(fig):
    y_mm = SUPP_TSNE_Y_MM
    for tsne_path in SUPP_TSNE_PATHS:
        tsne_image = plt.imread(tsne_path)
        image_height_px, image_width_px = tsne_image.shape[:2]
        display_height_mm = SUPP_TSNE_DISPLAY_W_MM * image_height_px / image_width_px
        ax = fig.add_axes(
            supp_rect(SUPP_TSNE_X_MM, y_mm, SUPP_TSNE_DISPLAY_W_MM, display_height_mm),
            zorder=0,
        )
        ax.imshow(tsne_image, aspect="equal", interpolation="none")
        ax.axis("off")
        y_mm += display_height_mm + SUPP_TSNE_GAP_MM



def supp_raster_asset(path, role, display_width_mm):
    image_width_px = plt.imread(path).shape[1]
    return {
        "path": path.as_posix(),
        "sha256": supp_sha256(path),
        "role": role,
        "display_width_mm": display_width_mm,
        "effective_dpi": image_width_px / (display_width_mm / 25.4),
    }


def make_figure2_supplementary():
    fig = plt.figure(figsize=(8.27, 11.69), facecolor="white")
    fig.text(
        12 / SUPP_PAGE_W_MM,
        285 / SUPP_PAGE_H_MM,
        "Supplementary Figure 2 | Frozen no-match UMAPs, pooled W-space t-SNE panels, and concordance",
        ha="left",
        va="bottom",
        fontsize=SUPP_LABEL_PT,
    )
    draw_supp_nomatch_umaps(fig)
    draw_supp_concordance(fig)
    draw_supp_tsne_panels(fig)
    return fig

FIG2_SUPP_PNG = _compiled_dir / "Figure_2_supplementary_panel.png"
FIG2_SUPP_PDF = _compiled_dir / "Figure_2_supplementary_panel.pdf"
FIG2_SUPP_SVG = _compiled_dir / "Figure_2_supplementary_panel.svg"
FIG2_SUPP_EXPORT_MANIFEST = _compiled_dir / "Figure_2_supplementary_panel_manifest.json"

figure2_supplementary = make_figure2_supplementary()
figure2_supplementary.savefig(FIG2_SUPP_PNG, format="png", dpi=OUTPUT_DPI, facecolor="white")
figure2_supplementary.savefig(FIG2_SUPP_PDF, format="pdf", dpi=OUTPUT_DPI, facecolor="white")
with mpl.rc_context({"svg.fonttype": "none", "svg.image_inline": True}):
    figure2_supplementary.savefig(FIG2_SUPP_SVG, format="svg", dpi=OUTPUT_DPI, facecolor="white")
supplementary_export = {
    "canvas_mm": [SUPP_PAGE_W_MM, SUPP_PAGE_H_MM],
    "dpi": OUTPUT_DPI,
    "layout": "Single A4 portrait page with compact 3x3 no-match UMAP grid above a lower two-column block: pooled concordance at left and two stacked pooled W-space t-SNE panels at right",
    "methods": [label for _, label in SUPP_UMAP_METHODS],
    "datasets": [DATASET_LABEL[dataset] for dataset in SUPP_DATASET_ORDER],
    "source_manifest": SUPP_MANIFEST_PATH.relative_to(Path.cwd()).as_posix(),
    "source_manifest_sha256": supp_sha256(SUPP_MANIFEST_PATH),
    "raster_assets": [
        {
            "path": path.relative_to(Path.cwd()).as_posix(),
            "sha256": supp_sha256(path),
            "role": "immutable frozen no-match UMAP point panels",
        }
        for path in SUPP_UMAP_FILES.values()
    ],
    "concordance_source": {"path": SUPP_CONCORDANCE_PATH.as_posix(), "sha256": supp_sha256(SUPP_CONCORDANCE_PATH), "rendering": "native vector rectangles and text"},
    "tsne_sources": [supp_raster_asset(path, "immutable pooled W-analysis t-SNE panel", SUPP_TSNE_DISPLAY_W_MM) for path in SUPP_TSNE_PATHS],
    "rasterization": "The nine immutable no-match UMAP panels and two existing pooled W-analysis t-SNE panels are rasterized; concordance, titles, legends, and surrounding text are vector.",
    "text_rendering": "Editable text in PDF/SVG using TrueType-compatible Matplotlib output.",
    "renderer": "matplotlib mixed raster/vector fallback",
    "matplotlib": mpl.__version__,
    "python": sys.version.split()[0],
    "svg_raster_images": sum(len(axis.images) for axis in figure2_supplementary.axes),
    "outputs": {
        fmt: {"path": path.relative_to(Path.cwd()).as_posix(), "sha256": supp_sha256(path)}
        for fmt, path in {"png": FIG2_SUPP_PNG, "pdf": FIG2_SUPP_PDF, "svg": FIG2_SUPP_SVG}.items()
    },
}
FIG2_SUPP_EXPORT_MANIFEST.write_text(json.dumps(supplementary_export, indent=2) + "\n")
plt.figure(figure2_supplementary.number)
plt.show()
print("Supplementary Figure 2 saved directly as PNG, PDF, and SVG.")
plt.close(figure2_supplementary)
